
# 00 — Data audit

What the two traces actually are, and the three measurement errors that shaped the
earlier results. Nothing here is modelling; it is all things that can be checked
directly against the CSVs.

The three findings, in order of how much they matter:

1. **Neither trace is hourly.** GP samples every ~86 min and Robi every ~99 min, so
   `lag_24` spans 34.4 h and 39.6 h — roughly anti-phase to the daily cycle.
2. **The rolling features leaked the target.** Windows were computed without a shift,
   putting `y_t` inside its own feature vector.
3. **The context flags act on variance, not level.** Most do not predict how much
   demand there will be; `is_rain` predicts how *wrong* the forecast will be.

In [ ]:
# --- Colab bootstrap -------------------------------------------------------
# Works in Colab and locally. In Colab, clone the repo first:
#     !git clone <repo-url> bwalloc && %cd bwalloc
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "src" / "bwalloc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

try:
    import xgboost  # noqa: F401
except ImportError:
    !pip install -q xgboost

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import bwalloc as bw
from bwalloc.plots import use_paper_style

bw.set_seed()
use_paper_style()
pd.set_option("display.width", 200)
RESULTS = ROOT / "experiments" / "results"
FIGURES = ROOT / "paper" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("bwalloc", bw.__version__, "| results:", RESULTS)

## Sampling rate

Every seasonal hyperparameter in this project is derived from this table. Nothing is hardcoded to 24.

In [ ]:

from bwalloc.data import load, sampling_profile, autocorrelation_by_lag

traces = {}
for operator in ("gp", "robi"):
    df = load(operator)
    profile = sampling_profile(df)
    traces[operator] = (df, profile)
    print(f"{operator.upper():5} {profile.describe()}")
    print(f"      lag 24 samples spans {profile.hours_for_lag(24):.1f} h "
          f"— the original code called this 'one day'")
    print(f"      one day is actually {profile.lag_for_hours(24)} samples\n")

## Autocorrelation: the lag the original models used is negatively correlated

This is the whole sampling-rate argument in one figure. The measured daily period has strong positive autocorrelation; lag 24 has negative.

In [ ]:

from bwalloc.plots import plot_autocorrelation_by_lag

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (operator, (df, profile)) in zip(axes, traces.items()):
    acf = autocorrelation_by_lag(df, max_lag=40)
    plot_autocorrelation_by_lag(acf, profile, ax=ax)
    ax.set_title(f"{operator.upper()} — {profile.median_gap_min:.0f} min/sample")
fig.tight_layout()
fig.savefig(FIGURES / "fig1_autocorrelation.png", dpi=200, bbox_inches="tight")

## The empirical consequence

A seasonal-naive forecaster at the measured period against the same forecaster at lag 24. Same data, same code, one hyperparameter.

In [ ]:

from bwalloc.metrics import rmse

rows = []
for operator, (df, profile) in traces.items():
    y = df["Gbps"]
    for lag, label in ((profile.daily_period, "measured daily period"), (24, "lag 24")):
        rows.append({
            "operator": operator, "lag": lag, "spans_hours": round(profile.hours_for_lag(lag), 1),
            "label": label, "rmse": rmse(y.iloc[lag:], y.shift(lag).dropna()),
        })
pd.DataFrame(rows)

## Target leakage

`assert_no_leakage` perturbs the final target value and asserts no feature column moves. The corrected builder passes; a builder with an unshifted rolling window would not.

In [ ]:

from bwalloc.features import FeatureConfig, assert_no_leakage, build_features

for operator, (df, profile) in traces.items():
    assert_no_leakage(df, profile, FeatureConfig())
    X, y = build_features(df, profile, FeatureConfig())
    print(f"{operator.upper():5} leak-free — {X.shape[1]} features, {len(X)} usable rows")

## Context flags — the dataset's distinctive asset, tested for the first time

Welch t-test on the level, Levene's test on the spread of hour-detrended residuals. The variance column is the one that matters: it is what the context-conditional method in notebook 03 is built on.

In [ ]:

from bwalloc.context import flag_report, group_sizes, assign_binary_groups

for operator, (df, profile) in traces.items():
    print(f"=== {operator.upper()} ===")
    report = flag_report(df, operator)
    display(report)
    print(group_sizes(assign_binary_groups(df)).to_string(index=False), "\n")


### Reading the table

Two things fall out, and both are used later:

- **`is_weekend` carries no signal at all** on the level (p = 0.77) — yet it is in
  every feature list, and *both source filenames advertise it*. Six of the nine flags
  are indistinguishable from noise on the mean.
- **`is_rain` raises residual σ from 16.4 to 21.8 Gbps — a 33% increase in
  uncertainty** — while moving the mean only modestly.

Context here acts on the **variance**, not the level. A point forecaster with a
context dummy cannot express that. A context-conditional *interval* can, and that is
notebook 03.